<a href="https://colab.research.google.com/github/AsutoshKumar28/Artificial_and_Computational_Intelligence_Assignment_2/blob/main/Question1_Group154.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIMLZG557: Artificial and Computational Intelligence
## Question 1: Human vs. AI Drop Token Game with Minimax (Group Submission)

**Task A: Static Evaluation Function Design and Justification**

**Evaluation Function Design**
The static evaluation function scores the board by sliding a "window" of 3 cells across all lines.
*   **3 AI tokens:** +100 points (Win)
*   **2 AI tokens + 1 empty space:** +10 points (Threat)
*   **1 AI token + 2 empty spaces:** +2 points (Building)
*   **3 Human tokens:** -100 points (Loss)
*   **2 Human tokens + 1 empty space:** -20 points (Opponent threat - prioritized for blocking)
*   **1 Human token + 2 empty spaces:** -2 points
*   **Mixed tokens:** 0 points

**Justification**
By assigning a higher penalty (-20) to the opponent having two tokens than the reward (+10) for the AI having two tokens, the heuristic forces the AI to prioritize defensive blocking before building its own non-winning lines.

In [ ]:
# Setup and Constants
import numpy as np
import math

ROW_COUNT = 5
COLUMN_COUNT = 5
WINDOW_LENGTH = 3

EMPTY = 0
HUMAN = 1
AI = 2

### Board Utility Functions
Handles the grid creation, token dropping, and win-condition checking.

In [ ]:
def create_board():
    return np.zeros((ROW_COUNT, COLUMN_COUNT), dtype=int)

def drop_piece(board, row, col, piece):
    board[row][col] = piece

def is_valid_location(board, col):
    return board[ROW_COUNT-1][col] == 0

def get_next_open_row(board, col):
    for r in range(ROW_COUNT):
        if board[r][col] == 0:
            return r

def print_board(board):
    print("\n" + str(np.flip(board, 0)))
    print("---------")
    print("0 1 2 3 4  <- Columns\n")

def winning_move(board, piece):
    for c in range(COLUMN_COUNT - 2):
        for r in range(ROW_COUNT):
            if board[r][c] == piece and board[r][c+1] == piece and board[r][c+2] == piece: return True
    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT - 2):
            if board[r][c] == piece and board[r+1][c] == piece and board[r+2][c] == piece: return True
    for c in range(COLUMN_COUNT - 2):
        for r in range(ROW_COUNT - 2):
            if board[r][c] == piece and board[r+1][c+1] == piece and board[r+2][c+2] == piece: return True
    for c in range(COLUMN_COUNT - 2):
        for r in range(2, ROW_COUNT):
            if board[r][c] == piece and board[r-1][c+1] == piece and board[r-2][c+2] == piece: return True
    return False

def get_valid_locations(board):
    return [col for col in range(COLUMN_COUNT) if is_valid_location(board, col)]

def is_terminal_node(board):
    return winning_move(board, HUMAN) or winning_move(board, AI) or len(get_valid_locations(board)) == 0

### Minimax Algorithm
Implements the lookahead tree search. Instead of printing every single node (which would flood the output with hundreds of lines), the interactivity is built into the main game loop to show top-level column evaluations.

In [ ]:
def evaluate_window(window, piece):
    score = 0
    opp_piece = HUMAN if piece == AI else AI

    if window.count(piece) == 3: score += 100
    elif window.count(piece) == 2 and window.count(EMPTY) == 1: score += 10
    elif window.count(piece) == 1 and window.count(EMPTY) == 2: score += 2

    if window.count(opp_piece) == 3: score -= 100
    elif window.count(opp_piece) == 2 and window.count(EMPTY) == 1: score -= 20
    elif window.count(opp_piece) == 1 and window.count(EMPTY) == 2: score -= 2

    return score

def score_position(board, piece):
    score = 0
    center_array = [int(i) for i in list(board[:, COLUMN_COUNT//2])]
    score += center_array.count(piece) * 3

    for r in range(ROW_COUNT):
        row_array = [int(i) for i in list(board[r,:])]
        for c in range(COLUMN_COUNT - 2):
            score += evaluate_window(row_array[c:c+WINDOW_LENGTH], piece)

    for c in range(COLUMN_COUNT):
        col_array = [int(i) for i in list(board[:,c])]
        for r in range(ROW_COUNT - 2):
            score += evaluate_window(col_array[r:r+WINDOW_LENGTH], piece)

    for r in range(ROW_COUNT - 2):
        for c in range(COLUMN_COUNT - 2):
            score += evaluate_window([board[r+i][c+i] for i in range(WINDOW_LENGTH)], piece)

    for r in range(ROW_COUNT - 2):
        for c in range(COLUMN_COUNT - 2):
            score += evaluate_window([board[r+2-i][c+i] for i in range(WINDOW_LENGTH)], piece)

    return score

def minimax(board, depth, maximizingPlayer):
    valid_locations = get_valid_locations(board)
    is_terminal = is_terminal_node(board)

    if depth == 0 or is_terminal:
        if is_terminal:
            if winning_move(board, AI): return (None, 100000000)
            elif winning_move(board, HUMAN): return (None, -100000000)
            else: return (None, 0)
        else:
            return (None, score_position(board, AI))

    if maximizingPlayer:
        value = -math.inf
        best_col = np.random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            b_copy = board.copy()
            drop_piece(b_copy, row, col, AI)
            new_score = minimax(b_copy, depth-1, False)[1]
            if new_score > value:
                value = new_score
                best_col = col
        return best_col, value
    else:
        value = math.inf
        best_col = np.random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            b_copy = board.copy()
            drop_piece(b_copy, row, col, HUMAN)
            new_score = minimax(b_copy, depth-1, True)[1]
            if new_score < value:
                value = new_score
                best_col = col
        return best_col, value

### Main Execution (Interactive Step-by-Step)
Run this block to play. The AI will output its evaluation score for every available column on its turn, showing exactly how it calculates its optimal move.

In [ ]:
def play_game():
    board = create_board()
    game_over = False
    turn = 0

    print("===========================================")
    print("  Welcome to 5x5 Drop Token (Target: 3)    ")
    print("===========================================")
    print_board(board)

    while not game_over:
        if turn == 0:
            try:
                col = int(input("Human (Token 1) - Choose a column (0-4): "))
            except ValueError:
                print("Invalid input. Please enter a number 0-4.")
                continue

            if col < 0 or col >= COLUMN_COUNT:
                print("Invalid column. Please choose 0-4.")
                continue

            if is_valid_location(board, col):
                row = get_next_open_row(board, col)
                print(f"\n[HUMAN ACTION] Dropping token in Column {col}, Row {row}")
                drop_piece(board, row, col, HUMAN)

                if winning_move(board, HUMAN):
                    print_board(board)
                    print("\n🎉 HUMAN WINS! 🎉")
                    game_over = True
            else:
                print("Column is full! Choose another one.")
                continue

        else:
            print("\n[AI TURN] Analyzing possible moves (Depth 3)...")
            valid_locations = get_valid_locations(board)
            best_score = -math.inf
            best_col = np.random.choice(valid_locations)

            # Interactive breakdown of AI's top-level choices
            for col in valid_locations:
                row = get_next_open_row(board, col)
                b_copy = board.copy()
                drop_piece(b_copy, row, col, AI)
                # Call minimax for depth-1 since we took the first step manually here
                score = minimax(b_copy, 2, False)[1]
                print(f" -> Testing Col {col} (Lands in Row {row}): Projected Minimax Score = {score}")

                if score > best_score:
                    best_score = score
                    best_col = col

            print(f">>> AI selects Column {best_col} (Highest Score: {best_score}) <<<")

            row = get_next_open_row(board, best_col)
            drop_piece(board, row, best_col, AI)

            if winning_move(board, AI):
                print_board(board)
                print("\n🤖 AI WINS! 🤖")
                game_over = True

        if not game_over and len(get_valid_locations(board)) == 0:
            print_board(board)
            print("\n🤝 IT'S A DRAW! 🤝")
            game_over = True

        if not game_over:
            print_board(board)

        turn += 1
        turn = turn % 2

play_game()

  Welcome to 5x5 Drop Token (Target: 3)    

[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
---------
0 1 2 3 4  <- Columns

Human (Token 1) - Choose a column (0-4): 3

[HUMAN ACTION] Dropping token in Column 3, Row 0

[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 0]]
---------
0 1 2 3 4  <- Columns


[AI TURN] Analyzing possible moves (Depth 3)...
 -> Testing Col 0 (Lands in Row 0): Projected Minimax Score = -22
 -> Testing Col 1 (Lands in Row 0): Projected Minimax Score = 1
 -> Testing Col 2 (Lands in Row 0): Projected Minimax Score = -2
 -> Testing Col 3 (Lands in Row 1): Projected Minimax Score = -5
 -> Testing Col 4 (Lands in Row 0): Projected Minimax Score = -2
>>> AI selects Column 1 (Highest Score: 1) <<<

[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 2 0 1 0]]
---------
0 1 2 3 4  <- Columns

Human (Token 1) - Choose a column (0-4): 4

[HUMAN ACTION] Dropping token in Column 4, Row 0

[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0